# 28 Error Handling

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Know <b>when</b> Spark errors happen (analysis time vs execution time), catch them precisely with PySpark's error classes, prevent common runtime failures with <code>try_*</code> functions, add logging and retries, and make a pipeline step fail loudly when it should.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Production pipelines fail: a source file is missing, a column is renamed, a value can't be parsed, a network call times out. A good pipeline fails <b>fast, clearly and safely</b>: it reports what went wrong, doesn't leave half-written data, and doesn't silently swallow errors. Lazy evaluation makes this tricky, because the line that <b>causes</b> a problem is often not the line that <b>raises</b> it.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Two kinds of Spark errors</b>
<ul><li><b>Analysis errors</b> happen when Spark <b>resolves</b> the plan: missing table or column, wrong types, bad SQL syntax. They raise <code>AnalysisException</code> (or <code>ParseException</code>). On Spark Connect (serverless) most are reported when the plan is first analysed, which is usually at the first action or schema access</li><li><b>Execution errors</b> happen while <b>running</b> the job on executors: a value that can't be cast under ANSI mode, division by zero, a corrupt file. They surface at the <b>action</b> (<code>count</code>, <code>write</code>, <code>show</code>)</li><li>All PySpark errors extend <code>PySparkException</code>, which gives a stable <b>error condition</b> (<code>getCondition()</code>, such as <code>UNRESOLVED_COLUMN.WITH_SUGGESTION</code>) and a <b>SQLSTATE</b> (<code>getSqlState()</code>). Match on those, not on message text</li></ul>
</div>

```
df = spark.read.table("sales")          # may fail now: table missing (analysis)
df2 = df.withColumn("x", col("amt") / col("qty"))   # lazy, nothing runs
df2.write.saveAsTable("out")            # fails HERE if qty = 0 under ANSI (execution)
```

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A nightly job loads partner files. One night the partner renames <code>amount</code> to <code>amt</code>. Without error handling, the job dies with a long stack trace and the on-call engineer has to dig. With good handling, a schema check runs first and stops the job with "expected column <code>amount</code> missing from partner file 2024-11-05", the alert links to the file, and nothing is written. The next night a different partner sends <code>N/A</code> in a numeric field. That is handled by <code>try_cast</code> plus a quarantine, without failing at all.
</div>

## Syntax

```python
from pyspark.errors import AnalysisException, PySparkException

try:
    df = spark.table("workspace.spark_basics.sales")
    df.select("amount").count()
except AnalysisException as e:
    if e.getCondition() == "TABLE_OR_VIEW_NOT_FOUND":
        ...                                  # handle a known case
    else:
        raise                                # never hide unexpected errors
except PySparkException as e:
    print(e.getCondition(), e.getSqlState())
    raise
```

## Setup

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a small orders DataFrame with a zero quantity and a bad amount.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
4 orders. Order 3 has quantity 0, order 4 has the amount <code>"N/A"</code>.
</div>

In [0]:
import logging
from pyspark.sql import functions as F
from pyspark.errors import AnalysisException, PySparkException

orders = spark.createDataFrame(
    [(1, "120.50", 2), (2, "80.00", 1), (3, "45.00", 0), (4, "N/A", 3)],
    "order_id INT, amount STRING, qty INT",
)
orders.show()

## 1. Analysis errors: wrong names and missing tables

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
References a column and a table that don't exist, catches <code>AnalysisException</code>, and reads the structured error details.<br><br>
<b>Why it matters</b><br><code>getCondition()</code> and <code>getSqlState()</code> are stable identifiers you can branch on. Error messages can change between versions, conditions don't. <code>getMessageParameters()</code> gives the details, such as the missing name and suggestions.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
For the column: <code>UNRESOLVED_COLUMN.WITH_SUGGESTION</code>, SQLSTATE <code>42703</code>, and parameters showing <code>amout</code> and the suggested columns. For the table: <code>TABLE_OR_VIEW_NOT_FOUND</code> and SQLSTATE <code>42P01</code>.
</div>

In [0]:
try:
    orders.select("amout").collect()
except AnalysisException as e:
    print("condition :", e.getCondition())
    print("sqlstate  :", e.getSqlState())
    print("details   :", e.getMessageParameters())

try:
    spark.table("no_such_table").count()
except AnalysisException as e:
    print("condition :", e.getCondition(), "| sqlstate:", e.getSqlState())

## 2. Execution errors appear at the action

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Defines a division and a cast that will fail under ANSI mode, shows that <b>defining</b> them raises nothing, and that the <b>action</b> raises the error.<br><br>
<b>Why it matters</b><br>Because of lazy evaluation, the <code>try</code> must wrap the <b>action</b>, not just the transformation. A <code>try</code> around <code>withColumn</code> alone catches nothing.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
"Defined without error", then a caught error with condition <code>DIVIDE_BY_ZERO</code> (SQLSTATE <code>22012</code>) at <code>collect()</code>. Then <code>CAST_INVALID_INPUT</code> for the amount.
</div>

In [0]:
try:
    per_unit = orders.withColumn("per_unit", F.col("amount").cast("double") / F.col("qty"))
    print("Defined without error (nothing has run yet)")
except PySparkException as e:
    print("never printed:", e)

try:
    per_unit.filter("order_id = 3").collect()          # the action triggers execution
except PySparkException as e:
    print("At the action:", e.getCondition(), "| sqlstate:", e.getSqlState())

try:
    orders.select(F.col("amount").cast("double")).collect()
except PySparkException as e:
    print("Cast failure:", e.getCondition())

## 3. Prevent errors in the data with `try_*` functions

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Rewrites the calculation with <code>try_cast</code> and <code>try_divide</code>, which return null instead of failing, and counts the problem rows.<br><br>
<b>Why it matters</b><br>Data problems in individual rows shouldn't kill a whole job. The <code>try_*</code> family (<code>try_cast</code>, <code>try_divide</code>, <code>try_to_timestamp</code>, <code>try_element_at</code>, <code>try_add</code>, ...) makes bad rows null, so you can <b>measure and quarantine</b> them. That's better than both crashing and silently ignoring them.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>per_unit</code> values for orders 1 and 2, null for order 3 (division by zero) and order 4 (bad amount), then <code>problem rows: 2</code>.
</div>

In [0]:
safe = orders.withColumns({
    "amount_num": F.col("amount").try_cast("double"),
    "per_unit": F.try_divide(F.col("amount").try_cast("double"), F.col("qty")),
})
safe.show()

problems = safe.filter(F.col("per_unit").isNull())
print("problem rows:", problems.count())

## 4. Validate early and fail with a clear message

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Checks that required columns exist and the input isn't empty <b>before</b> processing, and raises a clear <code>ValueError</code> otherwise.<br><br>
<b>Why it matters</b><br>A clear error at the start ("column <code>amount</code> is missing") is far better than an obscure one twenty steps later. Checking the schema costs nothing, because it reads metadata only.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The valid input passes. The input missing <code>qty</code> fails with <code>ValueError: missing required columns: ['qty']</code>.
</div>

In [0]:
def validate_input(df, required_columns, name="input"):
    """Fail fast with a readable message if the DataFrame can't be processed."""
    missing = [c for c in required_columns if c not in df.columns]
    if missing:
        raise ValueError(f"{name}: missing required columns: {missing}")
    if df.isEmpty():
        raise ValueError(f"{name}: no rows to process")
    return df

validate_input(orders, ["order_id", "amount", "qty"], "orders")
print("orders: OK")

try:
    validate_input(orders.drop("qty"), ["order_id", "amount", "qty"], "orders_v2")
except ValueError as e:
    print("ValueError:", e)

## 5. Logging instead of print

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Sets up a Python logger and logs the start, row counts, warnings and the end of a step.<br><br>
<b>Why it matters</b><br>Logs have levels (<code>INFO</code>, <code>WARNING</code>, <code>ERROR</code>), timestamps and a consistent format, and job logs keep them. <code>print</code> is fine for exploration, but pipelines should log. Log counts and decisions, not every row.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Log lines with a timestamp, level and message, including a WARNING about 2 problem rows.
</div>

In [0]:
import sys

logger = logging.getLogger("lesson28")
if not logger.handlers:                       # avoid duplicate handlers when re-running the cell
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(name)s | %(message)s"))
    logger.addHandler(handler)
logger.setLevel(logging.INFO)
logger.propagate = False

logger.info("starting step: clean_orders")
total, bad = safe.count(), problems.count()
logger.info("rows read: %d", total)
if bad:
    logger.warning("%d of %d rows could not be converted and will be quarantined", bad, total)
logger.info("finished step: clean_orders")

## 6. Retries for transient failures

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Wraps a flaky operation in a small retry helper with exponential backoff, and only retries errors that are worth retrying.<br><br>
<b>Why it matters</b><br>Network calls, API sources and some cloud storage operations fail <b>temporarily</b>. Retrying them makes a pipeline robust. But retrying a <b>logic</b> error (a missing column) just wastes time, so retry only transient errors. Jobs also have task-level retries, covered in <code>07_workflows</code>.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Two failed attempts with growing waits, then success on the third attempt. A non-retryable <code>ValueError</code> is raised immediately without retrying.
</div>

In [0]:
import time

class TransientError(Exception):
    """Stands in for a timeout or a temporary service error."""

def with_retries(func, attempts=3, base_delay=0.5, retry_on=(TransientError,)):
    for attempt in range(1, attempts + 1):
        try:
            return func()
        except retry_on as e:
            if attempt == attempts:
                logger.error("giving up after %d attempts: %s", attempts, e)
                raise
            delay = base_delay * 2 ** (attempt - 1)
            logger.warning("attempt %d failed (%s), retrying in %.1fs", attempt, e, delay)
            time.sleep(delay)

calls = {"n": 0}
def flaky_source():
    calls["n"] += 1
    if calls["n"] < 3:
        raise TransientError("service unavailable")
    return spark.range(5).count()

print("result:", with_retries(flaky_source))

try:
    with_retries(lambda: (_ for _ in ()).throw(ValueError("bad config")))
except ValueError as e:
    print("not retried:", e)

## 7. Putting it together: a safe pipeline step

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Combines validation, <code>try_*</code> conversion, quarantine, logging and a clear failure into one function that processes the orders.<br><br>
<b>Why it matters</b><br>This is the shape of a production step: check inputs, transform safely, separate bad rows, write the good ones, and <b>fail</b> (re-raise) if the bad-row rate passes a threshold, so problems are never silently ignored.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
With a 60% threshold, the step succeeds and returns 2 good and 2 quarantined rows. With a 10% threshold, it raises a <code>RuntimeError</code> because 50% of the rows are bad.
</div>

In [0]:
def clean_orders(df, max_bad_ratio=0.1):
    logger.info("clean_orders: start")
    validate_input(df, ["order_id", "amount", "qty"], "orders")
    converted = df.withColumns({
        "amount_num": F.col("amount").try_cast("double"),
        "per_unit": F.try_divide(F.col("amount").try_cast("double"), F.col("qty")),
    })
    good = converted.filter(F.col("per_unit").isNotNull())
    bad = converted.filter(F.col("per_unit").isNull()).withColumn("error_reason",
        F.when(F.col("amount_num").isNull(), "invalid amount").otherwise("zero quantity"))
    total, n_bad = converted.count(), bad.count()
    ratio = n_bad / total if total else 0
    logger.info("clean_orders: %d rows, %d bad (%.0f%%)", total, n_bad, ratio * 100)
    if ratio > max_bad_ratio:
        raise RuntimeError(f"clean_orders: bad-row ratio {ratio:.0%} exceeds limit {max_bad_ratio:.0%}")
    return good, bad

good, bad = clean_orders(orders, max_bad_ratio=0.6)
good.show(); bad.select("order_id", "amount", "qty", "error_reason").show()

try:
    clean_orders(orders, max_bad_ratio=0.1)
except RuntimeError as e:
    print("RuntimeError:", e)

## 8. Ending a notebook task with a status (Databricks)

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Shows how a notebook can report success or failure to the job that runs it.<br><br>
<b>Why it matters</b><br>In a job, an <b>uncaught exception fails the task</b> (and triggers retries and alerts). <code>dbutils.notebook.exit(value)</code> ends the notebook successfully and returns a value to the caller. Never catch an error just to call <code>exit</code> with a "success" status, because that hides failures from the job.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
<code>dbutils</code> exists only on Databricks. The <code>exit</code> line is commented out because it would stop the rest of this notebook. Try it at the end of a notebook run as a job task.
</div>

```python
try:
    good, bad = clean_orders(orders)
    # ... write the results ...
    # dbutils.notebook.exit(f"OK: {good.count()} rows written")
except Exception:
    logger.exception("pipeline step failed")
    raise          # re-raise so the job task is marked as FAILED
```

## Under the hood

```
   your code                      when it can fail
   -------------------------------------------------------------
   spark.table("t") / select(...)  analysis: names, types, syntax     -> AnalysisException
   withColumn / filter / join      nothing runs (lazy)
   count / show / write / collect  execution on executors             -> SparkException-family errors
                                     (cast, divide by zero, bad files,  e.g. ArithmeticException,
                                      out of memory, lost executors)        NumberFormatException

   On Spark Connect, the server returns the error class and SQLSTATE
   to the Python client, which raises the matching PySpark exception.
```

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Action</span> most data errors appear only at the action that runs the job.

<span style="background:#2F6FED;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Transformation</span> a `try` around transformations alone catches only analysis errors.

The `try_*` functions are ordinary expressions in the plan. Look for `try_divide` in the plan below: no special error handling machinery, just a null result.

In [0]:
spark.range(5).select(F.try_divide(F.lit(10), F.col("id")).alias("x")).explain()

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: <code>except Exception: pass</code></b><br>
Swallowing errors hides failures and corrupts data silently. Catch specific errors, log them, and re-raise what you can't handle.<br><br>
**⛔ Problem: the <code>try</code> block doesn't catch the error**<br>The error happens at the action, outside the <code>try</code>. Wrap the action (<code>count</code>, <code>write</code>), or the whole step.<br><br>
<b>⛔ Problem: matching on error message text</b><br>Messages change between versions. Use <code>e.getCondition()</code> and <code>e.getSqlState()</code>.<br><br>
<b>⛔ Problem: one bad value fails the whole job</b><br>ANSI mode turns bad casts into errors. Use <code>try_cast</code>, <code>try_divide</code> and friends, and quarantine the nulls they produce.<br><br>
<b>⛔ Problem: retries hammer a broken source for hours</b><br>Retry only transient errors, with a limit and backoff.<br><br>
<b>⛔ Problem: partial output after a failure</b><br>Write with Delta (atomic commits), and write as the <b>last</b> step after validation passes.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. When do errors occur in a Spark program?</b><br>
Analysis errors (missing tables or columns, type mismatches, syntax) occur when Spark resolves the plan. Execution errors (bad casts under ANSI, division by zero, corrupt files, out of memory) occur only when an action runs the job, because of lazy evaluation.<br><br>

<b>🎤 2. How do you catch a specific Spark error in PySpark?</b><br>
Catch <code>AnalysisException</code> or the base <code>PySparkException</code> from <code>pyspark.errors</code>, and branch on <code>e.getCondition()</code> (the error class) or <code>e.getSqlState()</code> instead of the message text.<br><br>

<b>🎤 3. How do you stop one bad row from failing a whole job?</b><br>
Use the <code>try_*</code> functions such as <code>try_cast</code> and <code>try_divide</code>, which return null instead of failing, then count and quarantine those rows. For files, use <code>PERMISSIVE</code> mode with a corrupt-record column.<br><br>

<b>🎤 4. Should a pipeline ever swallow exceptions?</b><br>
No. It should log the error with context and re-raise, so the job is marked failed and alerts fire. It can handle expected, recoverable cases explicitly, for example a retry of a transient error or quarantining bad rows below a threshold.<br><br>

<b>🎤 5. How do you implement retries correctly?</b><br>
Retry only transient failures (timeouts, throttling), with a maximum number of attempts and exponential backoff. Logic or data errors are not retried. On Databricks, job tasks also have built-in retry settings.<br><br>

<b>🎤 6. Why prefer logging over print in pipelines?</b><br>
Logs carry a timestamp, level and source, can be filtered and collected by the job's log system, and give consistent, searchable output.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A notebook defines <code>df2 = df.withColumn("ratio", col("a") / col("b"))</code> and the next line is <code>df2.write.saveAsTable("t")</code>. Some rows have <code>b = 0</code>, and ANSI mode is enabled. Where does the error surface?</b><br>
A. On the <code>withColumn</code> line<br>
B. On the <code>saveAsTable</code> line<br>
C. Never, division by zero always returns NULL<br>
D. When the notebook is opened<br>
<details><summary><b>Show answer</b></summary><b>B.</b> Transformations are lazy, so the error appears when the write (an action) executes. With ANSI off, it would return NULL instead.</details><br><br>

<b>Q2. Which expression returns NULL instead of an error when dividing by zero in Databricks SQL?</b><br>
A. <code>divide(a, b)</code><br>
B. <code>try_divide(a, b)</code><br>
C. <code>safe_divide(a, b)</code><br>
D. <code>a // b</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b> <code>a / nullif(b, 0)</code> is another valid pattern.</details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Write a function <code>read_table_or_empty(name, schema)</code> that returns the table if it exists, or an empty DataFrame with the given schema if the error condition is <code>TABLE_OR_VIEW_NOT_FOUND</code>, and re-raises any other error</li><li>Convert <code>orders.amount</code> with <code>try_cast</code>, and log a WARNING with the number of rows that failed</li><li>Write a validation that fails if any <code>order_id</code> is duplicated, with a clear message listing the duplicate IDs</li><li>Show that a <code>try</code> around <code>withColumn(... / F.col("qty"))</code> doesn't catch the division error, but a <code>try</code> around <code>.collect()</code> does</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Handle one known error, re-raise the rest
def read_table_or_empty(name, schema):
    try:
        df = spark.table(name)
        df.schema                      # force analysis so a missing table fails here
        return df
    except AnalysisException as e:
        if e.getCondition() == "TABLE_OR_VIEW_NOT_FOUND":
            logger.warning("table %s not found, using an empty DataFrame", name)
            return spark.createDataFrame([], schema)
        raise

print(read_table_or_empty("missing_table", "id INT, name STRING").count())

# 2. Log failed conversions
converted = orders.withColumn("amount_num", F.col("amount").try_cast("double"))
failed = converted.filter(F.col("amount").isNotNull() & F.col("amount_num").isNull()).count()
if failed:
    logger.warning("%d amount values could not be converted", failed)

# 3. Duplicate check with a clear message
def assert_unique(df, key):
    dupes = [r[key] for r in df.groupBy(key).count().filter("count > 1").collect()]
    if dupes:
        raise ValueError(f"duplicate {key} values: {dupes}")

assert_unique(orders, "order_id")
try:
    assert_unique(orders.union(orders.limit(1)), "order_id")
except ValueError as e:
    print("ValueError:", e)

# 4. Laziness: only the action raises
try:
    lazy = orders.withColumn("x", F.lit(1) / F.col("qty"))
    print("no error yet")
    lazy.collect()
except PySparkException as e:
    print("caught at collect():", e.getCondition())

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Analysis errors (names, types, syntax) vs execution errors (data problems at the action)</li><li>Lazy evaluation: wrap the <b>action</b> in <code>try</code>, not just the transformation</li><li>Catch <code>AnalysisException</code> / <code>PySparkException</code>, and branch on <code>getCondition()</code> and <code>getSqlState()</code></li><li>Prevent row-level failures with <code>try_cast</code>, <code>try_divide</code>, <code>try_to_timestamp</code>, and quarantine the nulls</li><li>Validate inputs first (columns, emptiness, keys) with clear messages</li><li>Log with levels, retry only transient errors with backoff, and <b>re-raise</b> what you can't handle</li><li>In jobs, an uncaught exception fails the task, which is what you want for real errors</li></ul>
</div>

| Task | Code |
|---|---|
| Catch analysis errors | `except AnalysisException as e:` |
| Error class | `e.getCondition()`, `e.getSqlState()`, `e.getMessageParameters()` |
| Safe cast | `F.col("x").try_cast("double")` |
| Safe divide | `F.try_divide(a, b)` |
| Empty check | `df.isEmpty()` |
| Logger | `logging.getLogger("name")`, `logger.warning(...)` |
| Fail clearly | `raise ValueError("what went wrong and where")` |

## Git commit

```
git add 01_spark_basics/28_error_handling.ipynb
git commit -m "add 28 error handling notebook"
```